In this program we will create facts & dimensions for modelling

In [3]:
import numpy as np
import pandas as pd
import os
import glob
from functools import reduce

folder_path = os.getcwd()
dataset_folder_path = os.path.join(folder_path, "..\\data_clean\\")


### Merging all indices data

In [18]:
# Get all CSV files
csv_files = glob.glob(os.path.join(dataset_folder_path, "*.csv"))

dfs = []

for file in csv_files:
    file_name = os.path.splitext(os.path.basename(file))[0]

    # Apply filter on filename only
    if file_name.startswith('nifty') or file_name == 'market_index':

        df = pd.read_csv(file)

        # Ensure required columns exist
        if 'date' not in df.columns or 'close' not in df.columns:
            continue

        df = df[['date', 'close']]

        df = df.rename(columns={'close': f'{file_name}_value'})

        dfs.append(df)

# Safety check
if not dfs:
    raise ValueError("No valid CSV files found after filtering.")

# Merge all dataframes on trade_date
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='date', how='outer'),
    dfs
)

# Convert to datetime and sort
merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d-%m-%Y')
merged_df = merged_df.sort_values('date')

# Save output
merged_df.to_csv(f"{dataset_folder_path}..\\data_model\\facts_indices.csv", index=False)

print(merged_df.head())


          date  market_index_value  nifty_auto_value  nifty_bank_value  \
0   2008-01-01             3220.99           2370.46           9905.25   
144 2008-01-02             3282.05           2392.41          10250.85   
283 2008-01-03             3284.43           2383.54          10100.70   
437 2008-01-04             3306.04           2376.58          10233.20   
894 2008-01-07             3330.91           2375.87          10391.10   

     nifty_energy_value  nifty_financial_services_value  nifty_fmcg_value  \
0                   NaN                         4329.07           6477.59   
144                 NaN                         4443.85           6513.67   
283                 NaN                         4396.72           6482.09   
437                 NaN                         4478.07           6603.97   
894                 NaN                         4523.48           6778.92   

     nifty_it_value  nifty_metal_value  nifty_pharma_value  nifty_realty_value  
0          

**Note:** Since our energy dataset is from Feb 2011 it will show NAN till there